# Credit Risk Starter Notebook

This notebook is for beginner-friendly exploration of the Home Credit data.
Use scripts in src/ for stable training and prediction runs.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
RAW_DIR

WindowsPath('c:/Users/DELL/Desktop/credit risk prediction/home-credit-default-risk/data/raw')

In [2]:
sorted([p.name for p in RAW_DIR.glob('*.csv')])

['HomeCredit_columns_description.csv',
 'POS_CASH_balance.csv',
 'application_test.csv',
 'application_train.csv',
 'bureau.csv',
 'bureau_balance.csv',
 'credit_card_balance.csv',
 'installments_payments.csv',
 'previous_application.csv',
 'sample_submission.csv']

In [3]:
train_df = pd.read_csv(RAW_DIR / 'application_train.csv')
test_df = pd.read_csv(RAW_DIR / 'application_test.csv')

train_df.shape, test_df.shape

((307511, 122), (48744, 121))

In [4]:
train_df[['TARGET']].value_counts(normalize=True).rename('ratio')

TARGET
0         0.919271
1         0.080729
Name: ratio, dtype: float64

In [5]:
train_df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## Step 2: Data quality report outputs

This section reads the generated report files from `reports/`.
If you have not created them yet, run:

```bash
python generate_data_report.py
```

In [ ]:
import json
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
REPORT_DIR = PROJECT_DIR / "reports"

report_path = REPORT_DIR / "data_quality_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    report["application_train_target"]
else:
    print("Report not found. Run: python generate_data_report.py")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

missingness_path = REPORT_DIR / "application_train_missingness.csv"
if missingness_path.exists():
    missingness_df = pd.read_csv(missingness_path)
    top_missing = missingness_df.head(20).copy()
    display(top_missing)

    plt.figure(figsize=(8, 6))
    plt.barh(top_missing["column"], top_missing["missing_percent"])
    plt.gca().invert_yaxis()
    plt.title("Top 20 Columns by Missing % (application_train)")
    plt.xlabel("Missing percent")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.show()
else:
    print("Missingness file not found. Run: python generate_data_report.py")

## Explainability report (SHAP)

This section reads the explainability outputs generated by:

`python generate_explainability_report.py`

It will show:
- the summary metadata from JSON
- the top features ranked by mean absolute SHAP value
- the SHAP summary bar chart image

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
REPORT_DIR = PROJECT_DIR / "reports"

summary_path = REPORT_DIR / "explainability_summary.json"
top_features_path = REPORT_DIR / "explainability_top_features.csv"
plot_path = REPORT_DIR / "figures" / "shap_summary_bar.png"

summary = json.loads(summary_path.read_text(encoding="utf-8"))
top_features_df = pd.read_csv(top_features_path)

print("Explainer model:", summary.get("explainer_model"))
print("Rows explained:", summary.get("rows_explained"))
print("Feature count:", summary.get("feature_count"))

print("\nTop 10 SHAP features")
display(top_features_df.head(10))

print("\nSHAP summary plot")
display(Image(filename=str(plot_path)))